# 03 · Embedding

**Belgedeki bölüm:** Embedding

- Embedding: token'ı, anlam taşıyan bir sayı vektörüne çevirmek. Token ID sadece bir etikettir, anlam taşımaz.
- Benzer anlamdaki kelimelerin vektörleri birbirine yakındır (mesafe = benzerlik, yön = ilişki).
- Değerler başta rastgeledir; eğitimde (loss + geri yayılım) öğrenilir. Eksenlerin adı yoktur (latent dimensions).

In [ ]:
import torch
from usta_tokenizer import UstaTokenizer

tokenizer = UstaTokenizer("tokenizer.json")

## Embedding yani Sözlük Anlamlarının Sayısal Değerleri
## Eng: Embedding is the numerical representation of the dictionary meanings

In [ ]:
the_1 = 0.1
the_2 = 0.4
the_3 = 0.0
the_4 = 0.0

capital_1 = 0.0
capital_2 = 0.2
capital_3 = 0.24
capital_4 = 0.3

of_1 = 0.10
of_2 = 0.04
of_3 = 0.02
of_4 = 0.01

united_1 = 0.30
united_2 = 0.06
united_3 = 0.03
united_4 = 0.01

In [ ]:
dict_v1 = [
  [0.1, 0.0, 0.0, 0.0],
  [0.4, 0.2, 0.04, 0.3],
  [0.0, 0.24, 0.02, 0.01],
  [0.0, 0.0, 0.03, 0.01],
  [0.0, 0.0, 0.0, 0.0]
]

dict_v1[0], dict_v1[1], dict_v1[2], dict_v1[3]

In [ ]:
dict_v2 = {
  "the": [0.1, 0.0, 0.0, 0.0],
  "capital": [0.4, 0.2, 0.04, 0.3],
  "of": [0.0, 0.24, 0.02, 0.01],
  "united": [0.0, 0.0, 0.03, 0.01],
  "states": [0.0, 0.0, 0.0, 0.0]
}

dict_v2["the"], dict_v2["capital"], dict_v2["of"], dict_v2["united"], dict_v2["states"]

In [ ]:
import plotly.graph_objects as go
import plotly.offline

def plot_dots(sentences_data, title, dims=[0, 1, 2]):
  data = [
    go.Scatter3d(
      x=sentence_data["words"][:, dims[0]],
      y=sentence_data["words"][:, dims[1]],
      z=sentence_data["words"][:, dims[2]],
      mode="markers+text",
      marker=dict(
        size=6,
        color=sentence_data["color"],
      ),
      text=sentence_data["labels"],
      hoverinfo="text",
    ) for sentence_data in sentences_data
  ]

  layout = go.Layout(
    scene=dict(
      xaxis_title="Sertlik",
      yaxis_title="Parlaklık",
      zaxis_title="Kırmızılık",
    ),
    title=title,
  )

  fig = go.Figure(data=data, layout=layout)
  plotly.offline.iplot(fig)

In [ ]:
import numpy as np

sentences = [
  {
    "words": np.array([
      [0.1, 0.20, 0.23, 0.0],
      [0.4, 0.2, 0.04, 0.3],
      [0.12, 0.24, 0.2, 0.01],
      [0.0, 0.0, 0.3, 0.01],
      [0.0, 0.0, 0.0, 0.0]
    ]),
    "labels": ["the", "capital", "of", "united", "states"],
    "color": "red",
  },
  {
    "words": np.array([
      [0.4, 0.2, 0.04, 0.3],
      [0.12, 0.24, 0.2, 0.01],
      [0.0, 0.0, 0.3, 0.01],
      [0.0, 0.0, 0.0, 0.0]
    ]),
    "labels": ["capital", "of", "united", "states"],
    "color": "blue",
  }
]

plot_dots(sentences, "Sözlük V1")

In [ ]:
import torch

embeddings = torch.nn.Embedding(num_embeddings=64, embedding_dim=4)

In [ ]:
sentence = "the capital of united states and the capital of france"

In [ ]:
tokens = tokenizer.encode(sentence)
tokens

In [ ]:
meanings = embeddings(tokens)
meanings

In [ ]:
sentences = [
  {
    "words": meanings.detach().numpy(),
    "labels": tokenizer.tokenize(sentence),
    "color": "red",
  },
]
plot_dots(sentences, "Sözlük V1")

## İsteğe bağlı: Gerçek bir modelin embedding'leri (Gemma 3 1B)
Model ~2 GB indirir ve Hugging Face girişi gerektirir.

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

login()

gemma_tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
gemma_model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it")

In [ ]:
gemma_tokens = gemma_tokenizer.encode(sentence)
gemma_tokens

In [ ]:
gemma_model.model.embed_tokens(torch.tensor(gemma_tokens))

Burada 11 Kelime var bu kelimelerim 1152 tane boyutta anlamları var

In [ ]:
gemma_meanings = gemma_model.model.embed_tokens(torch.tensor(gemma_tokens))
gemma_meanings.shape

In [ ]:
gemma_tokenizer.tokenize(sentence)

In [ ]:
gemma_sentences = [
  {
    "words": gemma_meanings.detach().float().numpy(),
    "labels": gemma_tokenizer.tokenize(sentence),
    "color": "red",
  },
]
plot_dots(gemma_sentences, "Gemma", dims=[20, 21, 22])